
# E-commerce Data Engineering Platform

## Business Scenario

We are building a data engineering platform for an e-commerce business.

The business generates data from orders, customers, products, payments,
reviews, sellers, and deliveries.

Currently, business data is difficult to analyze consistently.
Management needs reliable and regularly updated data to make decisions.

## Business Problems

- Sales performance is difficult to track.
- Order data may contain duplicates or missing values.
- Delivery performance is not easy to measure.
- Product performance reporting is inconsistent.
- Business reports require manual processing.

## Project Objectives

- Build a scalable data pipeline.
- Ingest raw data into the Bronze layer.
- Clean and validate data in the Silver layer.
- Create business-ready analytics in the Gold layer.
- Track invalid and rejected records.
- Automate data processing using Databricks Jobs.
- Implement incremental data processing.
- Produce reliable business KPIs.

## Technology Stack

- Databricks
- Python
- PySpark
- SQL
- Delta Lake
- Git and GitHub

## Architecture

Bronze -> Silver -> Gold

## Project Reload Cell — Copy & Save

In [0]:
# ==========================================
# PROJECT RELOAD / SETUP CELL
# Olist E-commerce Data Engineering Project
# ==========================================

from pyspark.sql.functions import col, sum, when, count, round

# Raw dataset location
raw_path = "/Volumes/workspace/ecommerce_de_project/raw_files/"

# Reload Orders dataset
orders_path = raw_path + "olist_orders_dataset.csv"

df_orders_raw = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(orders_path)
)

# Basic project variables
orders_count = df_orders_raw.count()

print("Project environment loaded successfully!")
print(f"Total Orders: {orders_count}")
print("Orders DataFrame: df_orders_raw")

Project environment loaded successfully!
Total Orders: 99441
Orders DataFrame: df_orders_raw


In [0]:
df_order_items_raw = spark.read.csv(
    raw_path + "olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

df_order_items_raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



## Create Poject Schema 

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_de_project
""")

DataFrame[]

## Create Volume

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce_de_project.raw_files
""")

DataFrame[]

## Path File Name AND Size Check

In [0]:
raw_path = "/Volumes/workspace/ecommerce_de_project/raw_files/"

files = dbutils.fs.ls(raw_path)

for file in files:
    size_mb = file.size / (1024 * 1024)

    print(f"File Name : {file.name}")
    print(f"File Size : {size_mb:.2f} MB")
    print("-" * 40)

File Name : olist_customers_dataset.csv
File Size : 8.62 MB
----------------------------------------
File Name : olist_geolocation_dataset.csv
File Size : 58.44 MB
----------------------------------------
File Name : olist_order_items_dataset.csv
File Size : 14.72 MB
----------------------------------------
File Name : olist_order_payments_dataset.csv
File Size : 5.51 MB
----------------------------------------
File Name : olist_order_reviews_dataset.csv
File Size : 13.78 MB
----------------------------------------
File Name : olist_orders_dataset.csv
File Size : 16.84 MB
----------------------------------------
File Name : olist_products_dataset.csv
File Size : 2.27 MB
----------------------------------------
File Name : olist_sellers_dataset.csv
File Size : 0.17 MB
----------------------------------------
File Name : product_category_name_translation.csv
File Size : 0.00 MB
----------------------------------------


In [0]:
file_report = []

for file in dbutils.fs.ls(raw_path):
    file_report.append({
        "file_name": file.name,
        "size_mb": round(file.size / (1024 * 1024), 2)
    })

display(spark.createDataFrame(file_report))

file_name,size_mb
olist_customers_dataset.csv,8.62
olist_geolocation_dataset.csv,58.44
olist_order_items_dataset.csv,14.72
olist_order_payments_dataset.csv,5.51
olist_order_reviews_dataset.csv,13.78
olist_orders_dataset.csv,16.84
olist_products_dataset.csv,2.27
olist_sellers_dataset.csv,0.17
product_category_name_translation.csv,0.0


## CSV Read করো

In [0]:
orders_path = raw_path + "olist_orders_dataset.csv"

df_orders_raw = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(orders_path)
)

print("Orders dataset loaded successfully")

Orders dataset loaded successfully


## Show Schema 

In [0]:
df_orders_raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



## Orders-এর Total Record Count বের করা

In [0]:
orders_count = df_orders_raw.count()

print(f"Total Orders Records: {orders_count}")

Total Orders Records: 99441


## Order Status Distribution Analysis
একটি E-commerce company-র জন্য শুধু total order জানলেই হয় না। Management জানতে চায়:\
- কত order delivered হয়েছে?\
- কত order canceled হয়েছে?\
- কত order processing অবস্থায় আছে?\
- কোন status-এ অস্বাভাবিক বেশি order রয়েছে?\
এটি আমাদের প্রথম Business Analysis using PySpark।

In [0]:
from pyspark.sql.functions import col

df_orders_raw.groupBy("order_status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



## Order Status Percentage বের করা

In [0]:
from pyspark.sql.functions import col, round, count

total_orders = df_orders_raw.count()

df_status_analysis = (
    df_orders_raw
    .groupBy("order_status")
    .agg(count("*").alias("total_orders"))
    .withColumn(
        "percentage",
        round(col("total_orders") / total_orders * 100, 2)
    )
    .orderBy(col("total_orders").desc())
)

display(df_status_analysis)

order_status,total_orders,percentage
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.3
created,5,0.01
approved,2,0.0


## Data Quality: NULL Value Analysis

In [0]:
from pyspark.sql.functions import col, sum, when

null_report = df_orders_raw.select(
    [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_orders_raw.columns
    ]
)

display(null_report)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,0,0,160,1783,2965,0


## NULL কেন হচ্ছে? Status অনুযায়ী Investigation

In [0]:
from pyspark.sql.functions import col, sum, when

df_null_by_status = (
    df_orders_raw
    .groupBy("order_status")
    .agg(
        sum(
            when(col("order_approved_at").isNull(), 1).otherwise(0)
        ).alias("missing_approval"),

        sum(
            when(col("order_delivered_carrier_date").isNull(), 1).otherwise(0)
        ).alias("missing_carrier_date"),

        sum(
            when(col("order_delivered_customer_date").isNull(), 1).otherwise(0)
        ).alias("missing_delivery_date")
    )
    .orderBy("order_status")
)

display(df_null_by_status)

order_status,missing_approval,missing_carrier_date,missing_delivery_date
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


## Delivered Order Must Have Delivery Timestamp

In [0]:
from pyspark.sql.functions import col

df_delivered_missing_date = (
    df_orders_raw
    .filter(
        (col("order_status") == "delivered") &
        (col("order_delivered_customer_date").isNull())
    )
)

print("Delivered orders with missing delivery date:")
print("Count:", df_delivered_missing_date.count())

display(df_delivered_missing_date)

Delivered orders with missing delivery date:
Count: 8


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28T17:44:07.000Z,2017-11-28T17:56:40.000Z,2017-11-30T18:12:23.000Z,null,2017-12-18T00:00:00.000Z
f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20T06:58:43.000Z,2018-06-20T07:19:05.000Z,2018-06-25T08:05:00.000Z,null,2018-07-16T00:00:00.000Z
2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01T17:05:11.000Z,2018-07-01T17:15:12.000Z,2018-07-03T13:57:00.000Z,null,2018-07-30T00:00:00.000Z
e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01T22:05:55.000Z,2018-07-01T22:15:14.000Z,2018-07-03T13:57:00.000Z,null,2018-07-30T00:00:00.000Z
0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01T21:14:02.000Z,2018-07-01T21:29:54.000Z,2018-07-03T09:28:00.000Z,null,2018-07-24T00:00:00.000Z
2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25T23:22:43.000Z,2017-05-25T23:30:16.000Z,null,null,2017-06-23T00:00:00.000Z
ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08T12:09:39.000Z,2018-06-08T12:36:39.000Z,2018-06-12T14:10:00.000Z,null,2018-06-26T00:00:00.000Z
20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27T16:09:12.000Z,2018-06-27T16:29:30.000Z,2018-07-03T19:26:00.000Z,null,2018-07-19T00:00:00.000Z


## Task 19: Find suspicious delivered orders

In [0]:
df_delivered_missing_date = (
    df_orders_raw
    .filter(
        (col("order_status") == "delivered") &
        (col("order_delivered_customer_date").isNull())
    )
)

print("Suspicious delivered orders:")
print("Count:", df_delivered_missing_date.count())

display(df_delivered_missing_date)

Suspicious delivered orders:
Count: 8


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28T17:44:07.000Z,2017-11-28T17:56:40.000Z,2017-11-30T18:12:23.000Z,null,2017-12-18T00:00:00.000Z
f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20T06:58:43.000Z,2018-06-20T07:19:05.000Z,2018-06-25T08:05:00.000Z,null,2018-07-16T00:00:00.000Z
2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01T17:05:11.000Z,2018-07-01T17:15:12.000Z,2018-07-03T13:57:00.000Z,null,2018-07-30T00:00:00.000Z
e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01T22:05:55.000Z,2018-07-01T22:15:14.000Z,2018-07-03T13:57:00.000Z,null,2018-07-30T00:00:00.000Z
0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01T21:14:02.000Z,2018-07-01T21:29:54.000Z,2018-07-03T09:28:00.000Z,null,2018-07-24T00:00:00.000Z
2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25T23:22:43.000Z,2017-05-25T23:30:16.000Z,null,null,2017-06-23T00:00:00.000Z
ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08T12:09:39.000Z,2018-06-08T12:36:39.000Z,2018-06-12T14:10:00.000Z,null,2018-06-26T00:00:00.000Z
20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27T16:09:12.000Z,2018-06-27T16:29:30.000Z,2018-07-03T19:26:00.000Z,null,2018-07-19T00:00:00.000Z


## Inspect the 8 Suspicious Records

In [0]:
from pyspark.sql.functions import col

df_delivered_missing_date.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(8, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|ec05a6d8558c6455f0cbbd8a420ad34f|delivered   |2017-11-28 17:44:07     |2017-11-28 17:56:40|2017-11-30 18:12:23         |NULL                         |2017-12-18 00:00:00          |
|f5dd62b788049ad9fc0526e3ad11a097|5e89028e024b381dc84a13a3570decb4|delivered   |2018-06-20 06:58:43     |2018-06-20 07:19:05|2018-06-25 08:0

## Create a Data Quality Flag

In [0]:
from pyspark.sql.functions import when, lit, col

df_orders_flagged = (
    df_orders_raw
    .withColumn(
        "dq_status",
        when(
            (col("order_status") == "delivered") &
            (col("order_delivered_customer_date").isNull()),
            lit("INVALID_CANDIDATE")
        ).otherwise(lit("VALID"))
    )
    .withColumn(
        "dq_issue",
        when(
            col("dq_status") == "INVALID_CANDIDATE",
            lit("DELIVERED_MISSING_DELIVERY_DATE")
        ).otherwise(lit(None))
    )
)

print("Total records:", df_orders_flagged.count())
print("Invalid candidates:", df_orders_flagged.filter(col("dq_status") == "INVALID_CANDIDATE").count())

display(
    df_orders_flagged
    .filter(col("dq_status") == "INVALID_CANDIDATE")
    .select("order_id", "order_status", "order_delivered_customer_date", "dq_status", "dq_issue")
)

Total records: 99441
Invalid candidates: 8


order_id,order_status,order_delivered_customer_date,dq_status,dq_issue
2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
f5dd62b788049ad9fc0526e3ad11a097,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
2ebdfc4f15f23b91474edf87475f108e,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
e69f75a717d64fc5ecdfae42b2e8e086,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
0d3268bad9b086af767785e3f0fc0133,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
2d858f451373b04fb5c984a1cc2defaf,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE
20edc82cf5400ce95e1afacc25798b31,delivered,null,INVALID_CANDIDATE,DELIVERED_MISSING_DELIVERY_DATE


## Valid Records ও Rejected Candidates আলাদা করা

In [0]:
# Separate valid records
df_orders_valid = (
    df_orders_flagged
    .filter(col("dq_status") == "VALID")
)

# Separate rejected candidates
df_orders_rejected = (
    df_orders_flagged
    .filter(col("dq_status") == "INVALID_CANDIDATE")
)

print("Valid records:", df_orders_valid.count())
print("Rejected candidates:", df_orders_rejected.count())
print("Total:", df_orders_valid.count() + df_orders_rejected.count())

Valid records: 99433
Rejected candidates: 8
Total: 99441


## Create Your First Bronze Delta Table 🥉

In [0]:
# Create Bronze Delta table from original raw orders

bronze_table_name = "workspace.ecommerce_de_project.bronze_orders"

(
    df_orders_raw
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table_name)
)

print("Bronze table created successfully!")

Bronze table created successfully!


## Bronze Table Verify (SQL)

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.bronze_orders
LIMIT 10;

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


## Bronze Record Count Validation

In [0]:
%sql
SELECT
    COUNT(*) AS bronze_total_records
FROM workspace.ecommerce_de_project.bronze_orders;

bronze_total_records
99441


## Bronze Data Profiling (Professional ETL)

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,

    SUM(CASE WHEN order_id IS NULL
        THEN 1 ELSE 0 END) AS missing_order_id,

    SUM(CASE WHEN customer_id IS NULL
        THEN 1 ELSE 0 END) AS missing_customer_id,

    SUM(CASE WHEN order_status IS NULL
        THEN 1 ELSE 0 END) AS missing_order_status,

    SUM(CASE WHEN order_purchase_timestamp IS NULL
        THEN 1 ELSE 0 END) AS missing_purchase_timestamp

FROM workspace.ecommerce_de_project.bronze_orders;

total_records,missing_order_id,missing_customer_id,missing_order_status,missing_purchase_timestamp
99441,0,0,0,0


## Duplicate Order Detection

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS duplicate_count
FROM workspace.ecommerce_de_project.bronze_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

order_id,duplicate_count


## Define Silver Layer Rules 🥈

### Silver Cleaning Rules Test

In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN order_id IS NULL
              OR customer_id IS NULL
              OR order_status IS NULL
              OR order_purchase_timestamp IS NULL
            THEN 1
            ELSE 0
        END
    ) AS missing_critical_records,

    SUM(
        CASE
            WHEN order_status = 'delivered'
             AND order_delivered_customer_date IS NULL
            THEN 1
            ELSE 0
        END
    ) AS delivered_missing_date

FROM workspace.ecommerce_de_project.bronze_orders;

total_orders,missing_critical_records,delivered_missing_date
99441,0,8


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.bronze_order_items
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/ecommerce_de_project/raw_files/olist_order_items_dataset.csv',
    format => 'csv',
    header => true
);

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_order_items
FROM workspace.ecommerce_de_project.bronze_order_items;

total_order_items
112650


In [0]:
%sql

DESCRIBE workspace.ecommerce_de_project.bronze_order_items;

col_name,data_type,comment
order_id,string,null
order_item_id,int,null
product_id,string,null
seller_id,string,null
shipping_limit_date,timestamp,null
price,double,null
freight_value,double,null
_rescued_data,string,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,

    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_order_id,

    SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS missing_order_item_id,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,

    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS missing_seller_id,

    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price,

    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS missing_freight,

    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS invalid_price,

    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS negative_freight

FROM workspace.ecommerce_de_project.bronze_order_items;

total_records,missing_order_id,missing_order_item_id,missing_product_id,missing_seller_id,missing_price,missing_freight,invalid_price,negative_freight
112650,0,0,0,0,0,0,0,0


## Duplicate Order Items Check

In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    COUNT(*) AS duplicate_count

FROM workspace.ecommerce_de_project.bronze_order_items

GROUP BY
    order_id,
    order_item_id

HAVING COUNT(*) > 1

ORDER BY duplicate_count DESC;

order_id,order_item_id,duplicate_count


## Price & Freight Validation

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,

    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS invalid_price,

    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS negative_freight,

    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,

    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight,

    MIN(price) AS minimum_price,

    MAX(price) AS maximum_price,

    MIN(freight_value) AS minimum_freight,

    MAX(freight_value) AS maximum_freight

FROM workspace.ecommerce_de_project.bronze_order_items;

total_records,invalid_price,negative_freight,null_price,null_freight,minimum_price,maximum_price,minimum_freight,maximum_freight
112650,0,0,0,0,0.85,6735.0,0.0,409.68


## Create Quarantine Table

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.quarantine_order_items AS

SELECT
    *,
    CASE
        WHEN order_id IS NULL THEN 'MISSING_ORDER_ID'
        WHEN order_item_id IS NULL THEN 'MISSING_ORDER_ITEM_ID'
        WHEN product_id IS NULL THEN 'MISSING_PRODUCT_ID'
        WHEN seller_id IS NULL THEN 'MISSING_SELLER_ID'
        WHEN price IS NULL OR price <= 0 THEN 'INVALID_PRICE'
        WHEN freight_value IS NULL OR freight_value < 0 THEN 'INVALID_FREIGHT'
    END AS rejection_reason

FROM workspace.ecommerce_de_project.bronze_order_items

WHERE
    order_id IS NULL
    OR order_item_id IS NULL
    OR product_id IS NULL
    OR seller_id IS NULL
    OR price IS NULL
    OR price <= 0
    OR freight_value IS NULL
    OR freight_value < 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS rejected_records

FROM workspace.ecommerce_de_project.quarantine_order_items;

rejected_records
0


## Create Silver Orders Table 🥈

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.ecommerce_de_project.silver_orders AS

SELECT *
FROM workspace.ecommerce_de_project.bronze_orders

WHERE order_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND order_status IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL

  AND NOT (
      order_status = 'delivered'
      AND order_delivered_customer_date IS NULL
  );

num_affected_rows,num_inserted_rows


## Silver Record Count Validation

In [0]:
%sql

SELECT
  (SELECT COUNT(*) FROM workspace.ecommerce_de_project.bronze_orders) AS bronze_count,
  (SELECT COUNT(*) FROM workspace.ecommerce_de_project.silver_orders) AS silver_count,
  (SELECT COUNT(*) FROM workspace.ecommerce_de_project.bronze_orders) - (SELECT COUNT(*) FROM workspace.ecommerce_de_project.silver_orders) AS excluded_records;

bronze_count,silver_count,excluded_records
99441,99433,8


## Create Silver Order Items

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.silver_order_items AS

SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value

FROM workspace.ecommerce_de_project.bronze_order_items

WHERE
    order_id IS NOT NULL
    AND order_item_id IS NOT NULL
    AND product_id IS NOT NULL
    AND seller_id IS NOT NULL
    AND price IS NOT NULL
    AND price > 0
    AND freight_value IS NOT NULL
    AND freight_value >= 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS silver_total_records

FROM workspace.ecommerce_de_project.silver_order_items;

silver_total_records
112650


## Order Items clean করে Silver তৈরি

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.silver_order_items AS

SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    CAST(price AS DOUBLE) AS price,
    CAST(freight_value AS DOUBLE) AS freight_value

FROM workspace.ecommerce_de_project.bronze_order_items

WHERE
    order_id IS NOT NULL
    AND order_item_id IS NOT NULL
    AND product_id IS NOT NULL
    AND seller_id IS NOT NULL
    AND price IS NOT NULL
    AND freight_value IS NOT NULL
    AND price >= 0
    AND freight_value >= 0;

## Bronze vs Silver Reconciliation

In [0]:
%sql

SELECT

    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.bronze_order_items)
        AS bronze_count,

    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.silver_order_items)
        AS silver_count,

    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.quarantine_order_items)
        AS quarantine_count,

    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.bronze_order_items)
    -
    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.silver_order_items)
    -
    (SELECT COUNT(*)
     FROM workspace.ecommerce_de_project.quarantine_order_items)
        AS unreconciled_records;

bronze_count,silver_count,quarantine_count,unreconciled_records
112650,112650,0,0


## Silver Order Items Duplicate Validation

In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    COUNT(*) AS duplicate_count
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

order_id,order_item_id,duplicate_count


## Silver Order Items Data Quality Validation

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(order_id IS NULL) AS null_order_id,
    COUNT_IF(order_item_id IS NULL) AS null_order_item_id,
    COUNT_IF(product_id IS NULL) AS null_product_id,
    COUNT_IF(seller_id IS NULL) AS null_seller_id,
    COUNT_IF(price < 0) AS negative_price,
    COUNT_IF(freight_value < 0) AS negative_freight
FROM workspace.ecommerce_de_project.silver_order_items;

total_rows,null_order_id,null_order_item_id,null_product_id,null_seller_id,negative_price,negative_freight
112650,0,0,0,0,0,0


এই ধাপটি বাস্তব Data Engineer interview-তেও গুরুত্বপূর্ণ। Interviewer জিজ্ঞেস করতে পারে 
How do you ensure that no records are lost during data transformation?

তোমার উত্তর হবে:
I perform data reconciliation between the Bronze and Silver layers by comparing source records, validated records, and quarantined records. I also verify that the unreconciled record count is zero.

কোডটি রান করিয়ে screenshot পাঠাও। 💪

## Create Permanent Rejected / Quarantine Table

### Quarantine Table তৈরি করো

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.ecommerce_de_project.quarantine_orders AS

SELECT *,
       'DQ-ORD-001' AS rejection_rule,
       'Delivered order missing customer delivery date' AS rejection_reason,
       current_timestamp() AS rejected_at

FROM workspace.ecommerce_de_project.bronze_orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS quarantine_records,
    rejection_rule,
    rejection_reason
FROM workspace.ecommerce_de_project.quarantine_orders
GROUP BY rejection_rule, rejection_reason;

quarantine_records,rejection_rule,rejection_reason
8,DQ-ORD-001,Delivered order missing customer delivery date


## Build a Data Quality Monitoring Table

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.data_quality_monitoring AS

SELECT
    'BRONZE_TOTAL_RECORDS' AS metric_name,
    COUNT(*) AS metric_value,
    CURRENT_TIMESTAMP() AS checked_at
FROM workspace.ecommerce_de_project.bronze_orders

UNION ALL

SELECT
    'SILVER_TOTAL_RECORDS',
    COUNT(*),
    CURRENT_TIMESTAMP()
FROM workspace.ecommerce_de_project.silver_orders

UNION ALL

SELECT
    'QUARANTINE_TOTAL_RECORDS',
    COUNT(*),
    CURRENT_TIMESTAMP()
FROM workspace.ecommerce_de_project.quarantine_orders

UNION ALL

SELECT
    'MISSING_CRITICAL_FIELDS',
    COUNT(*),
    CURRENT_TIMESTAMP()
FROM workspace.ecommerce_de_project.bronze_orders
WHERE order_id IS NULL
   OR customer_id IS NULL
   OR order_status IS NULL
   OR order_purchase_timestamp IS NULL

UNION ALL

SELECT
    'DUPLICATE_ORDER_IDS',
    COUNT(*),
    CURRENT_TIMESTAMP()
FROM (
    SELECT order_id
    FROM workspace.ecommerce_de_project.bronze_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
);

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.data_quality_monitoring
ORDER BY metric_name;

metric_name,metric_value,checked_at
BRONZE_TOTAL_RECORDS,99441,2026-11-24T09:50:31.357Z
DUPLICATE_ORDER_IDS,0,2026-11-24T09:50:31.357Z
MISSING_CRITICAL_FIELDS,0,2026-11-24T09:50:31.357Z
QUARANTINE_TOTAL_RECORDS,8,2026-11-24T09:50:31.357Z
SILVER_TOTAL_RECORDS,99433,2026-11-24T09:50:31.357Z


## ### Data Quality PASS / FAIL System

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.data_quality_status AS

SELECT
    metric_name,
    metric_value,

    CASE
        WHEN metric_name = 'BRONZE_TOTAL_RECORDS'
             AND metric_value > 0
        THEN 'PASS'

        WHEN metric_name = 'SILVER_TOTAL_RECORDS'
             AND metric_value > 0
        THEN 'PASS'

        WHEN metric_name = 'QUARANTINE_TOTAL_RECORDS'
        THEN 'PASS'

        WHEN metric_name = 'MISSING_CRITICAL_FIELDS'
             AND metric_value = 0
        THEN 'PASS'

        WHEN metric_name = 'DUPLICATE_ORDER_IDS'
             AND metric_value = 0
        THEN 'PASS'

        ELSE 'FAIL'
    END AS quality_status,

    CURRENT_TIMESTAMP() AS evaluated_at

FROM workspace.ecommerce_de_project.data_quality_monitoring;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.data_quality_status
ORDER BY metric_name;

metric_name,metric_value,quality_status,evaluated_at
BRONZE_TOTAL_RECORDS,99441,PASS,2026-11-24T09:53:21.211Z
DUPLICATE_ORDER_IDS,0,PASS,2026-11-24T09:53:21.211Z
MISSING_CRITICAL_FIELDS,0,PASS,2026-11-24T09:53:21.211Z
QUARANTINE_TOTAL_RECORDS,8,PASS,2026-11-24T09:53:21.211Z
SILVER_TOTAL_RECORDS,99433,PASS,2026-11-24T09:53:21.211Z


## Overall Pipeline Health

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.pipeline_health_summary AS

SELECT
    COUNT(*) AS total_checks,

    SUM(CASE
        WHEN quality_status = 'PASS' THEN 1
        ELSE 0
    END) AS passed_checks,

    SUM(CASE
        WHEN quality_status = 'FAIL' THEN 1
        ELSE 0
    END) AS failed_checks,

    CASE
        WHEN SUM(CASE
            WHEN quality_status = 'FAIL' THEN 1
            ELSE 0
        END) = 0
        THEN 'HEALTHY'
        ELSE 'FAILED'
    END AS pipeline_status,

    CURRENT_TIMESTAMP() AS evaluated_at

FROM workspace.ecommerce_de_project.data_quality_status;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.pipeline_health_summary;

total_checks,passed_checks,failed_checks,pipeline_status,evaluated_at
5,5,0,HEALTHY,2026-11-24T09:55:42.922Z


## Monthly Order Analysis

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.ecommerce_de_project.gold_monthly_orders AS

SELECT
    DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS order_month,
    COUNT(*) AS total_orders

FROM workspace.ecommerce_de_project.silver_orders

GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')

ORDER BY order_month;

num_affected_rows,num_inserted_rows


## Gold Table Verify

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.gold_monthly_orders
ORDER BY order_month;

order_month,total_orders
2016-09,4
2016-10,324
2016-12,1
2017-01,800
2017-02,1780
2017-03,2682
2017-04,2404
2017-05,3699
2017-06,3245
2017-07,4026


## Best Performing Month বের করি

In [0]:
%sql

SELECT
    order_month,
    total_orders

FROM workspace.ecommerce_de_project.gold_monthly_orders

ORDER BY total_orders DESC

LIMIT 5;

order_month,total_orders
2017-11,7543
2018-01,7269
2018-03,7211
2018-04,6939
2018-05,6873


## Gold Monthly Revenue Table

In [0]:
%sql

SHOW TABLES IN workspace.ecommerce_de_project;

database,tableName,isTemporary
ecommerce_de_project,bronze_orders,false
ecommerce_de_project,data_quality_monitoring,false
ecommerce_de_project,data_quality_status,false
ecommerce_de_project,gold_monthly_orders,false
ecommerce_de_project,pipeline_health_summary,false
ecommerce_de_project,quarantine_orders,false
ecommerce_de_project,silver_orders,false


In [0]:
%sql
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS sales_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(i.order_item_id) AS total_items_sold,
    ROUND(SUM(i.price), 2) AS product_revenue,
    ROUND(SUM(i.freight_value), 2) AS freight_revenue,
    ROUND(SUM(i.price + i.freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_orders o
JOIN workspace.ecommerce_de_project.silver_order_items i
    ON o.order_id = i.order_id
GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
ORDER BY sales_month;

sales_month,total_orders,total_items_sold,product_revenue,freight_revenue,total_revenue
2016-09,3,6,267.36,87.39,354.75
2016-10,308,363,49507.66,7301.18,56808.84
2016-12,1,1,10.9,8.72,19.62
2017-01,789,955,120312.87,16875.62,137188.49
2017-02,1733,1951,247303.02,38977.6,286280.62
2017-03,2641,3000,374344.3,57704.29,432048.59
2017-04,2391,2684,359927.23,52495.01,412422.24
2017-05,3659,4135,505892.14,80104.81,585996.95
2017-06,3217,3583,433038.6,69924.44,502963.04
2017-07,3969,4519,498031.48,86940.14,584971.62


In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_de_project;

database,tableName,isTemporary
ecommerce_de_project,bronze_order_items,false
ecommerce_de_project,bronze_orders,false
ecommerce_de_project,data_quality_monitoring,false
ecommerce_de_project,data_quality_status,false
ecommerce_de_project,gold_monthly_orders,false
ecommerce_de_project,pipeline_health_summary,false
ecommerce_de_project,quarantine_order_items,false
ecommerce_de_project,quarantine_orders,false
ecommerce_de_project,silver_order_items,false
ecommerce_de_project,silver_orders,false


## Create Gold Monthly Revenue Table

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_monthly_revenue AS
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS sales_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(i.order_item_id) AS total_items_sold,
    ROUND(SUM(i.price), 2) AS product_revenue,
    ROUND(SUM(i.freight_value), 2) AS freight_revenue,
    ROUND(SUM(i.price + i.freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_orders o
JOIN workspace.ecommerce_de_project.silver_order_items i
    ON o.order_id = i.order_id
GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM');

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.gold_monthly_revenue
ORDER BY sales_month;

sales_month,total_orders,total_items_sold,product_revenue,freight_revenue,total_revenue
2016-09,3,6,267.36,87.39,354.75
2016-10,308,363,49507.66,7301.18,56808.84
2016-12,1,1,10.9,8.72,19.62
2017-01,789,955,120312.87,16875.62,137188.49
2017-02,1733,1951,247303.02,38977.6,286280.62
2017-03,2641,3000,374344.3,57704.29,432048.59
2017-04,2391,2684,359927.23,52495.01,412422.24
2017-05,3659,4135,505892.14,80104.81,585996.95
2017-06,3217,3583,433038.6,69924.44,502963.04
2017-07,3969,4519,498031.48,86940.14,584971.62


## Gold Product Sales Analytics

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS total_items_sold,
    ROUND(SUM(price), 2) AS product_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY product_id
ORDER BY product_revenue DESC
LIMIT 20;

product_id,total_items_sold,product_revenue,total_freight,total_revenue
bb50f2e236e5eea0100680137654686c,195,63885.0,3721.1,67606.1
6cdd53843498f92890544667809f1595,156,54730.2,4363.79,59093.99
d6160fb7873f184099d9bc95e30376af,35,48899.34,1426.84,50326.18
d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,13761.52,60976.03
99a4788cb24856965c36a24e339b6058,488,43025.56,8046.04,51071.6
3dd2a17168ec895c781a9191c1e95ad7,274,41082.6,7129.62,48212.22
25c38557cf793876c5abdd5931f922db,38,38907.32,1404.63,40311.95
5f504b3a1c75b73d6151be81eb05bdc9,63,37733.9,3991.91,41725.81
53b36df67ebb7c41585e8d54d6772e08,323,37683.42,2274.51,39957.93
aca2eb7d00ea1a7b8ebd4e68314663af,527,37608.9,7211.86,44820.76


## gold_top_products Table তৈরি

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_top_products AS
SELECT
    product_id,
    COUNT(*) AS total_items_sold,
    ROUND(SUM(price), 2) AS product_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY product_id
ORDER BY product_revenue DESC;

num_affected_rows,num_inserted_rows


## Gold Seller Performance Analytics

In [0]:
%sql
SELECT
    seller_id,
    COUNT(*) AS total_items_sold,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS total_product_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY seller_id
ORDER BY total_product_revenue DESC
LIMIT 20;

seller_id,total_items_sold,total_orders,total_product_revenue,total_freight,total_revenue
4869f7a5dfa277a7dca6462dcf3b52b2,1156,1132,229472.63,20168.07,249640.7
53243585a1d6dc2643021fd1853d8905,410,358,222776.05,13080.63,235856.68
4a3ca9315b744ce9f8e9374361493884,1987,1806,200472.92,35067.04,235539.96
fa1c13f2614d7b5c4749cbc52fecda94,586,585,194042.03,10042.7,204084.73
7c67e1448b00f6e969d365cea6b010ab,1364,982,187923.89,51612.55,239536.44
7e93a43ef30c4f03f38b393420bc753a,340,336,176431.87,6322.18,182754.05
da8622b14eb17ae2831f4ac5b9dab84a,1551,1314,160236.57,24955.75,185192.32
7a67c85e85bb2ce8582c35f2203ad736,1171,1160,141745.53,20902.85,162648.38
1025f0e2d44d7041d6cf58b6550e0bfa,1428,915,138968.55,33892.14,172860.69
955fee9216a65b617aa5c0531780ce60,1499,1287,135171.7,25430.98,160602.68


## Create Gold Seller Analytics Table

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_seller_performance AS
SELECT
    seller_id,
    COUNT(*) AS total_items_sold,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price), 2) AS total_product_revenue,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY seller_id;

num_affected_rows,num_inserted_rows


## Freight / Shipping Analytics

In [0]:
%sql
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS sales_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(i.freight_value), 2) AS total_freight,
    ROUND(AVG(i.freight_value), 2) AS avg_freight_per_item,
    ROUND(SUM(i.freight_value) / COUNT(DISTINCT o.order_id), 2) AS freight_per_order,
    ROUND(
        SUM(i.freight_value) / SUM(i.price) * 100,
        2
    ) AS freight_percentage_of_product_revenue
FROM workspace.ecommerce_de_project.silver_orders o
JOIN workspace.ecommerce_de_project.silver_order_items i
    ON o.order_id = i.order_id
GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
ORDER BY total_freight DESC;

sales_month,total_orders,total_freight,avg_freight_per_item,freight_per_order,freight_percentage_of_product_revenue
2018-03,7188,171913.38,20.92,23.92,17.48
2017-11,7450,168854.87,19.49,22.67,16.72
2018-07,6270,163167.04,23.02,26.02,18.23
2018-04,6934,163050.29,20.45,23.51,16.36
2018-06,6157,157509.36,22.26,25.58,18.22
2018-01,7220,157271.53,19.16,21.78,16.55
2018-05,6853,153264.14,19.34,22.36,15.38
2018-08,6452,148622.14,20.51,23.04,17.39
2018-02,6694,142730.25,18.6,21.32,16.91
2017-12,5624,119633.06,18.97,21.27,16.08


## Save Freight Analytics as Gold Table

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_monthly_freight AS
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS sales_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(i.freight_value), 2) AS total_freight,
    ROUND(AVG(i.freight_value), 2) AS avg_freight_per_item,
    ROUND(SUM(i.freight_value) / COUNT(DISTINCT o.order_id), 2) AS freight_per_order,
    ROUND(
        SUM(i.freight_value) / SUM(i.price) * 100,
        2
    ) AS freight_percentage_of_product_revenue
FROM workspace.ecommerce_de_project.silver_orders o
JOIN workspace.ecommerce_de_project.silver_order_items i
    ON o.order_id = i.order_id
GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM');

num_affected_rows,num_inserted_rows


## Order Status Analytics

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage_of_orders
FROM workspace.ecommerce_de_project.silver_orders
GROUP BY order_status
ORDER BY total_orders DESC;

order_status,total_orders,percentage_of_orders
delivered,96470,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


## Create Gold Order Status Table

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_order_status AS
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_orders
FROM workspace.ecommerce_de_project.silver_orders
GROUP BY order_status;

num_affected_rows,num_inserted_rows


##Check Schema

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_de_project;

database,tableName,isTemporary
ecommerce_de_project,bronze_order_items,false
ecommerce_de_project,bronze_orders,false
ecommerce_de_project,data_quality_monitoring,false
ecommerce_de_project,data_quality_status,false
ecommerce_de_project,gold_monthly_freight,false
ecommerce_de_project,gold_monthly_orders,false
ecommerce_de_project,gold_monthly_revenue,false
ecommerce_de_project,gold_order_status,false
ecommerce_de_project,gold_seller_performance,false
ecommerce_de_project,gold_top_products,false


## Business KPI Consolidation

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_de_project.gold_monthly_revenue;

col_name,data_type,comment
sales_month,string,null
total_orders,bigint,null
total_items_sold,bigint,null
product_revenue,double,null
freight_revenue,double,null
total_revenue,double,null
# Clustering Information,,
# col_name,data_type,comment


## Create Consolidated Monthly KPI Table

In [0]:
%sql
CREATE TABLE workspace.ecommerce_de_project.gold_monthly_business_kpis AS
SELECT
    sales_month,
    total_orders,
    total_items_sold,
    product_revenue,
    freight_revenue,
    total_revenue,
    ROUND(total_revenue / total_orders, 2) AS average_order_value
FROM workspace.ecommerce_de_project.gold_monthly_revenue;

num_affected_rows,num_inserted_rows


## Verify KPI Table

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.gold_monthly_business_kpis
ORDER BY sales_month;

sales_month,total_orders,total_items_sold,product_revenue,freight_revenue,total_revenue,average_order_value
2016-09,3,6,267.36,87.39,354.75,118.25
2016-10,308,363,49507.66,7301.18,56808.84,184.44
2016-12,1,1,10.9,8.72,19.62,19.62
2017-01,789,955,120312.87,16875.62,137188.49,173.88
2017-02,1733,1951,247303.02,38977.6,286280.62,165.19
2017-03,2641,3000,374344.3,57704.29,432048.59,163.59
2017-04,2391,2684,359927.23,52495.01,412422.24,172.49
2017-05,3659,4135,505892.14,80104.81,585996.95,160.15
2017-06,3217,3583,433038.6,69924.44,502963.04,156.35
2017-07,3969,4519,498031.48,86940.14,584971.62,147.39
